<a href="https://colab.research.google.com/github/purple1231/deepLearningStudy/blob/main/5%EC%A3%BC%EC%B0%A8_2143088_%EA%B9%80%EC%A7%84%EA%B7%BC.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

##**[5주차]과제**
- 아래의 과제1)의 코드를 완성하시오.
- 모든 코드의 결과를 출력하여, .ipynb의 링크를 **[5주차]/[5주차]과제**에 제출하시오.\
(실습 제출 예시: 5주차_2020XXXX_이름.ipynb 코드 링크)

In [1]:
print("2143088", "김진근")

2143088 김진근


In [2]:
from google.colab import auth

auth.authenticate_user()
!gcloud config get-value account

chop3930@gmail.com


In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
workspace_path = '/content/drive/MyDrive/deep/week_5/'

# [과제 1] activation func, hyper-parameter, optimizer등을 변경하여 와인 품질(quality)를 예측하는 모델의 accuracy(정확도)를 0.5 이상의 성능을 달성하여라.

## 조건 :
1. epoch 수는 최대 20으로 제한
2. 모델의 레이어는 2개(입력층, 출력층)로 제한
3. 입력층의 노드수는 256개로 하고 input_shape은 xTrain.shape[1]과 동일한 값으로 설정할 것

    예시) layer추가
```python
# 입력층 예시
netwrok.add(tf.keras.layers.Dense(노드 수, activation_func='activation_func', input_shape=xTrain.shape[1], kernel_initializer=initializer))
# 출력층 예시
network.add(tf.keras.layers.Dense(노드 수, activation='activation_func', kernel_initializer=initializer))

##- 개선사항을 작성해 주세요 :

L2 규제와 드롭아웃을 적용했습니다.
이는 가중치가 비정상적으로 커지는 것을 막고 일부 노드에만 의존하는 현상을 방지하여 검증 데이터에 대한 정확도를 높입니다.

그리고 softmax, Adam 옵티마이저를 통해 20 efforc 내에 0.5이상의 정확도를 내는 대에 성공했습니다.




In [5]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
import tensorflow as tf
from sklearn.model_selection import train_test_split

In [6]:
red_df = pd.read_csv(os.path.join(workspace_path,'winequality-red.csv'), sep=';', header=0, engine='python')
red_df.head()

,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,quality
0,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,5
1,7.8,0.88,0.00,2.6,0.098,25.0,67.0,0.9968,3.20,0.68,9.8,5
2,7.8,0.76,0.04,2.3,0.092,15.0,54.0,0.9970,3.26,0.65,9.8,5
3,11.2,0.28,0.56,1.9,0.075,17.0,60.0,0.9980,3.16,0.58,9.8,6
4,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,5


In [7]:
white_df = pd.read_csv(os.path.join(workspace_path,'winequality-white.csv'), sep=';', header=0, engine='python')
white_df.head()

,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,quality
0,7.0,0.27,0.36,20.7,0.045,45.0,170.0,1.0010,3.00,0.45,8.8,6
1,6.3,0.30,0.34,1.6,0.049,14.0,132.0,0.9940,3.30,0.49,9.5,6
2,8.1,0.28,0.40,6.9,0.050,30.0,97.0,0.9951,3.26,0.44,10.1,6
3,7.2,0.23,0.32,8.5,0.058,47.0,186.0,0.9956,3.19,0.40,9.9,6
4,7.2,0.23,0.32,8.5,0.058,47.0,186.0,0.9956,3.19,0.40,9.9,6


In [8]:
wine =  pd.concat([red_df, white_df])

In [9]:
wine.info()

<class 'pandas.core.frame.DataFrame'>
Index: 6497 entries, 0 to 4897
Data columns (total 12 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   fixed acidity         6497 non-null   float64
 1   volatile acidity      6497 non-null   float64
 2   citric acid           6497 non-null   float64
 3   residual sugar        6497 non-null   float64
 4   chlorides             6497 non-null   float64
 5   free sulfur dioxide   6497 non-null   float64
 6   total sulfur dioxide  6497 non-null   float64
 7   density               6497 non-null   float64
 8   pH                    6497 non-null   float64
 9   sulphates             6497 non-null   float64
 10  alcohol               6497 non-null   float64
 11  quality               6497 non-null   int64  
dtypes: float64(11), int64(1)
memory usage: 659.9 KB


In [10]:
sorted(wine.quality.unique())

[np.int64(3),
 np.int64(4),
 np.int64(5),
 np.int64(6),
 np.int64(7),
 np.int64(8),
 np.int64(9)]

In [11]:
wine.quality.value_counts()

,count
quality,
6,2836
5,2138
7,1079
4,216
8,193
3,30
9,5


In [12]:
wine.quality = wine.quality.apply(lambda x: x-3)
#레이블링 인코딩 모델의 출력층은 보통 0부터 시작하기 떄문에 3~9에서 -3을 한다.

In [13]:
sorted(wine.quality.unique())
# [np.int64(3),
#  np.int64(4),
#  np.int64(5),
#  np.int64(6),
#  np.int64(7),
#  np.int64(8),
#  np.int64(9)]
#before

[np.int64(0),
 np.int64(1),
 np.int64(2),
 np.int64(3),
 np.int64(4),
 np.int64(5),
 np.int64(6)]

In [14]:
xTrain = wine.drop('quality', axis=1)
yTrain = wine['quality']

In [15]:
xTrain, xTest, yTrain, yTest = train_test_split(xTrain, yTrain, test_size=0.2)
#test_size=0.2: 전체 데이터 중 20%를 테스트용으로 사용하겠다는 의미.

In [16]:
# 실험 결과 재현을 위한 seed 고정
SEED = 4
os.environ['PYTHONHASHSEED'] = str(SEED)
os.environ['TF_DETERMINISTIC_OPS'] = '1'
tf.random.set_seed(SEED)
initializer = tf.keras.initializers.GlorotUniform(seed=SEED)

In [17]:
# 모델 작성
def build_model():
  network = tf.keras.models.Sequential()

  # 입력층: 노드 256개, L2 규제 추가
  #특정 레이어의 가중치(Weight) 자체에 직접 제약을 거는 설정값.
  network.add(tf.keras.layers.Dense(256,
                                   activation='relu',
                                   input_shape=(xTrain.shape[1],),
                                   kernel_initializer=initializer,
                                   kernel_regularizer=tf.keras.regularizers.l2(0.001)))

  # 드롭아웃 추가: 50%의 노드를 무작위로 제외하여 과적합 방지
  #특정 레이어의 출력을 다음 레이어로 전달할 때 신호를 무작위로 차단하는 필터 역할을 ㅓ하낟.
  network.add(tf.keras.layers.Dropout(0.3))

  # 출력층: 품질 클래스 7개 (0~6)
  network.add(tf.keras.layers.Dense(7, activation='softmax', kernel_initializer=initializer))

  # 옵티마이저 설정
  network.compile(optimizer='adam',
                  loss='sparse_categorical_crossentropy',
                  metrics=['accuracy'])

  return network

In [18]:
model = build_model()

history = model.fit(xTrain, yTrain, epochs=20, batch_size=32, validation_split=0.25)

Epoch 1/20


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


122/122 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - accuracy: 0.3380 - loss: 5.2799 - val_accuracy: 0.4377 - val_loss: 1.8698
Epoch 2/20
122/122 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.3528 - loss: 2.9018 - val_accuracy: 0.4446 - val_loss: 1.4214
Epoch 3/20
122/122 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.3762 - loss: 1.8180 - val_accuracy: 0.4592 - val_loss: 1.3502
Epoch 4/20
122/122 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.4126 - loss: 1.4474 - val_accuracy: 0.4608 - val_loss: 1.2816
Epoch 5/20
122/122 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.4375 - loss: 1.3214 - val_accuracy: 0.4700 - val_loss: 1.2422
Epoch 6/20
122/122 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.4483 - loss: 1.2985 - val_accuracy: 0.4677 - val_loss: 1.2300
Epoch 7/20
122/122 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - accuracy: 0.4616 - loss: 1.2594 - val_accuracy: 0.4600 - val_loss: 1.2208
Epoch 8/20
122/122 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.4681 - loss: 1.2524 - val_accuracy: 0.4600 - val_

In [19]:
test_loss, test_acc = model.evaluate(xTest, yTest)

41/41 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.5038 - loss: 1.1120


In [20]:
print('테스트 정확도 : ', round(test_acc,3))

테스트 정확도 :  0.504
